# SmartCart Day 5b - Gradio Demo

The finale: a **self-checkout demo**. Upload a counter photo, the app detects products, recognizes each via the ONNX head, finds catalog matches, and totals the basket from a simple price table - the whole week's work behind one button.

In [1]:
# 1) Runtime setup
# This course runs in Google Colab. Run this cell first.
# Install only the packages used in this notebook.
%pip install -q timm ultralytics onnxruntime gradio psutil

import os

# Drive stores the small cross-day bundle, not the image dataset.
#from google.colab import drive
#drive.mount('/content/drive')
#BUNDLE_DIR = '/content/drive/MyDrive/SmartCart'
BUNDLE_DIR = './data'



Note: you may need to restart the kernel to use updated packages.


### Embedded toolkit

This cell defines the helper functions used below. Run it once after setup.

In [2]:
from __future__ import annotations
import json
import os
import pathlib
import subprocess
import numpy as np
import pandas as pd
HERE = pathlib.Path.cwd()  # embedded in-notebook: no __file__, anchor on the working dir
GROCERY_DATASET_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'

class Bundle:
    """Small Google Drive folder that carries artifacts from one day to the next."""

    def __init__(self, root: str):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}

    def put_table(self, name, df: pd.DataFrame):
        df.to_csv(self.root / name, index=False)
        self._note(name)

    def get_table(self, name) -> pd.DataFrame:
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr: np.ndarray):
        np.save(self.root / name, arr)
        self._note(name)

    def get_array(self, name) -> np.ndarray:
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self

def open_bundle(drive_dir='/content/drive/MyDrive/SmartCart') -> Bundle:
    """Open the cross-day Drive bundle. If it is new, start with an empty manifest."""
    return Bundle(drive_dir).load()

def save_bundle(b: Bundle):
    b.save()
    print(f'[bundle] saved -> {b.root}')

def load_backbone(name='vit_small_patch14_dinov2.lvd142m', device='cpu'):
    """Frozen DINOv2-small feature extractor."""
    import timm
    m = timm.create_model(name, pretrained=True, num_classes=0, dynamic_img_size=True).eval().to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def extract_features(model, batches, device=None) -> np.ndarray:
    """Run image batches through a frozen backbone and return numpy features."""
    import torch
    if device is None:
        try:
            device = next(model.parameters()).device
        except (AttributeError, StopIteration):
            device = 'cpu'
    outs = []
    with torch.no_grad():
        for xb in batches:
            y = model(xb.to(device))
            outs.append(y.detach().cpu().numpy().astype('float32'))
    return np.concatenate(outs, 0)

class EmbeddingIndex:
    """Cosine nearest-neighbor index over catalog/gallery embeddings."""

    def __init__(self):
        self._nn = None
        self.meta = []

    def build(self, feats: np.ndarray, meta: list[dict]):
        from sklearn.neighbors import NearestNeighbors
        f = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-08)
        self._nn = NearestNeighbors(n_neighbors=min(10, len(f)), metric='cosine').fit(f)
        self.meta = meta
        self._feats = f
        return self

    def query(self, q: np.ndarray, k=5):
        qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-08)
        dist, idx = self._nn.kneighbors(qn, n_neighbors=min(k, len(self.meta)))
        return [[{**self.meta[j], 'score': float(1 - d)} for d, j in zip(drow, irow)] for drow, irow in zip(dist, idx)]

    def save(self, b: Bundle, prefix='gallery'):
        b.put_array(f'{prefix}_index.npy', self._feats)
        b.put_table(f'{prefix}_meta.csv', pd.DataFrame(self.meta))

    @classmethod
    def load(cls, b: Bundle, prefix='gallery'):
        feats = b.get_array(f'{prefix}_index.npy')
        meta = b.get_table(f'{prefix}_meta.csv').to_dict('records')
        return cls().build(feats, meta)

def basket_total(items, prices, default=0.0):
    return float(sum((prices.get(it['fine'], default) for it in items)))
# --- helpers are now available as plain functions/classes in this notebook ---
print('SmartCart toolkit ready')


SmartCart toolkit ready


In [3]:
# 2) Load the cross-day bundle
# The bundle stores artifacts we create during the week: labels, indexes, weights, ONNX files.
# It is NOT the image dataset. Images are loaded in the next data-source cell.
b = open_bundle(BUNDLE_DIR)
print('bundle:', b.root)
print('artifacts:', list(b.manifest.get('artifacts', {})))


bundle: data
artifacts: ['gallery_index.npy', 'gallery_meta.csv', 'catalog_prices.csv', 'labels.csv', 'sample_scene.jpg', 'detector.pt', 'crops_manifest.csv', 'head.pt', 'per_class_metrics.csv', 'error_report.md', 'detector_synthetic_baseline.pt', 'head.onnx', 'crops_manifest_real.csv', 'head_v2.pt', 'lift_table.csv']


## Load model + catalog prices

**What:** Open an ONNX session for the head, load the gallery, load `catalog_prices.csv`, and load the detector + backbone.

**Why:** These are the exact runtime pieces the app needs - no training, just serving.

**Watch for:** Prices are editable demo business data; the ONNX head replaces PyTorch at serve time.

In [4]:
from pathlib import Path
import onnxruntime as ort
import pandas as pd
import torch
# Load serving-time artifacts: ONNX head, gallery, price catalog, backbone, detector.
sess = ort.InferenceSession(str(Path(b.root)/'head.onnx'))
idx = EmbeddingIndex.load(b)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)
model = load_backbone(device=device)
classes = b.manifest.get('class_list') or sorted({m['fine'] for m in idx.meta})
price_catalog = b.get_table('catalog_prices.csv')
prices = dict(zip(price_catalog.fine, price_catalog.unit_price))
currency = price_catalog.currency.iloc[0] if 'currency' in price_catalog else 'USD'
from ultralytics import YOLO
det = YOLO(str(Path(b.root)/'detector.pt'))
print('serving', len(classes), 'classes with', len(prices), 'catalog prices in', currency)


using device: cuda
serving 81 classes with 82 catalog prices in USD


## Inference function

**What:** Define `detect_and_classify(image)`: detect -> crop -> embed -> ONNX argmax, returning the annotated image plus each item's crop and predicted label (no auto-totaling). Define `append_correction(bundle_root, crop_image, fine_true)` to persist a manually corrected crop + true label for retraining.

**Why:** These functions are the app's brain; the per-item correction UI (next) wraps them so a shopper can fix any misread label before checkout, rather than trusting the model blindly.

**Watch for:** Returns per-item crops/predictions instead of a pre-totaled basket - the UI computes the running total and final receipt only after corrections are applied.

In [5]:
from pathlib import Path
import torch, torchvision.transforms.v2 as T
from PIL import Image, ImageDraw
import numpy as np
import pandas as pd
import uuid
# Shared image preprocessing for DINOv2.
TF = T.Compose([T.ToImage(), T.Resize((224,224)), T.ToDtype(torch.float32, scale=True),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def feats_of_imgs(imgs, model):
    """Embed PIL crops without saving them to disk."""
    xs = torch.stack([TF(im.convert('RGB')) for im in imgs])
    return extract_features(model, [xs])

def detect_and_classify(image):
    """Detect products in `image`, classify each crop, and return per-item data for the correction UI."""
    img = image.convert('RGB')
    arr = np.array(img)
    result = det(arr, imgsz=640, conf=0.25, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    if len(boxes) == 0:
        boxes = np.array([[0, 0, img.width, img.height]], dtype='float32')
    crops, preds = [], []
    draw_img = img.copy()
    draw = ImageDraw.Draw(draw_img)
    for x1, y1, x2, y2 in boxes:
        crop = Image.fromarray(arr[int(y1):int(y2), int(x1):int(x2)])
        f = feats_of_imgs([crop], model)
        logits = sess.run(None, {'feat': f.astype('float32')})[0]
        label = classes[int(logits.argmax(1)[0])]
        crops.append(crop)
        preds.append(label)
        draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
        draw.text((x1, y1), label, fill='red')
    return draw_img, crops, preds

def append_correction(bundle_root, crop_image, fine_true):
    """Save a corrected crop image + its true label into the bundle's corrections manifest."""
    corrections_dir = Path(bundle_root) / 'crops_corrections'
    corrections_dir.mkdir(parents=True, exist_ok=True)
    crop_path = corrections_dir / f'correction_{uuid.uuid4().hex}.jpg'
    crop_image.convert('RGB').save(crop_path)
    manifest_path = Path(bundle_root) / 'corrections_manifest.csv'
    row = pd.DataFrame([{'crop_path': str(crop_path), 'fine_true': fine_true}])
    if manifest_path.exists():
        row = pd.concat([pd.read_csv(manifest_path), row], ignore_index=True)
    row.to_csv(manifest_path, index=False)
    return str(crop_path)

sample_path = Path(b.root) / 'sample_scene.jpg'
if sample_path.exists():
    _img, _crops, _preds = detect_and_classify(Image.open(sample_path))
    print(f'self-check: detect_and_classify found {len(_crops)} item(s) on the sample scene, e.g. {_preds[:3]}')
else:
    print('self-check skipped: no sample_scene.jpg in the bundle yet')

self-check: detect_and_classify found 1 item(s) on the sample scene, e.g. ['Sweet-Potato']


## Launch the app

**What:** Show a two-column checkout UI (photo + detection on the left; per-item thumbnail, product dropdown, and an editable price field on the right, styled with Gradio's `Soft` theme). Each photo's detected item(s) populate the item slots; clicking **Add to Cart** pushes whatever's currently shown into a running cart (shown in its own table below), so you can upload several separate photos and build up one checkout. **Confirm & Checkout** finalizes the whole cart into the receipt and clears it; **New Cart** clears everything to start over. Then launches the Gradio interface on port 7861 (separate from the original demo's 7860, so both can run side by side for comparison).

**Why:** This variant adds price editing alongside product relabeling - both gated by the same staff PIN, and a genuine price edit permanently updates `catalog_prices.csv`, not just this one receipt. Accumulating items via an explicit Add to Cart action (rather than trying to grow the same item slots across uploads) keeps each button click's update small and reliable.

**Watch for:** Set SC_LAUNCH_GRADIO=1 (e.g. in a separate cell above this one) to launch; CI just prints a hint. The cell also frees port 7861 automatically before launching, in case a previous run's server didn't shut down cleanly.


In [ ]:
import gradio as gr
import pandas as pd
from pathlib import Path

MAX_ITEMS = 6  # generous ceiling for items detected in ONE photo
STAFF_PIN = '1234'  # course-project stand-in for a real staff credential

def resolve_checkout_changes(preds, proposed_labels, proposed_prices, prices, authorized):
    """Decide final labels/prices per item, given proposed edits and staff authorization.

    price_changed is evaluated against the catalog price for the CURRENTLY selected
    (possibly corrected) label, not the original prediction - so a label correction's
    own auto-sync of the price field is never itself treated as a manual price override.
    An unauthorized attempted change (label or price) reverts BOTH to the original.
    """
    final_labels, final_prices, label_changed, price_changed, rejected = [], [], [], [], []
    for original, prop_label, prop_price in zip(preds, proposed_labels, proposed_prices):
        l_changed = bool(prop_label) and prop_label != original
        current_label = prop_label if prop_label else original
        catalog_price_for_current = prices.get(current_label, 1.0)
        p_changed = prop_price is not None and abs(float(prop_price) - float(catalog_price_for_current)) > 1e-9
        attempted = l_changed or p_changed
        if attempted and not authorized:
            final_labels.append(original)
            final_prices.append(prices.get(original, 1.0))
            label_changed.append(False)
            price_changed.append(False)
            rejected.append(True)
        else:
            final_labels.append(current_label)
            final_prices.append(float(prop_price) if p_changed else catalog_price_for_current)
            label_changed.append(l_changed)
            price_changed.append(p_changed)
            rejected.append(False)
    return final_labels, final_prices, label_changed, price_changed, rejected

def update_catalog_price(catalog_df, label, new_price):
    """Return a copy of catalog_df with `label`'s unit_price set to new_price (adds a row if missing)."""
    df = catalog_df.copy()
    mask = df.fine == label
    if mask.any():
        df.loc[mask, 'unit_price'] = new_price
    else:
        new_row = pd.DataFrame([{'fine': label, 'coarse': 'New', 'unit_price': new_price, 'currency': 'USD', 'unit': 'each'}])
        df = pd.concat([df, new_row], ignore_index=True)
    return df

def persist_catalog_price(bundle_root, label, new_price):
    """Update catalog_prices.csv on disk for `label`, returning the updated DataFrame."""
    catalog_path = Path(bundle_root) / 'catalog_prices.csv'
    catalog_df = pd.read_csv(catalog_path)
    catalog_df = update_catalog_price(catalog_df, label, new_price)
    catalog_df.to_csv(catalog_path, index=False)
    return catalog_df

def sync_price(label):
    """Refresh a price field to the newly-selected product's catalog price."""
    return gr.update(value=prices.get(label, 1.0)) if label else gr.update()

def format_items_markdown(rows):
    """Render a list of {'item': ..., 'price': ...} dicts as a markdown table.

    Used instead of gr.Dataframe for the cart/receipt displays: gr.Dataframe's SECOND
    value update within a session was found to not re-render (first update always
    worked, later ones silently failed to show on screen, even with an explicit
    headers/datatype/col_count schema) - a markdown string has no such issue.
    """
    if not rows:
        return '_(empty)_'
    lines = ['| item | price |', '|---|---|']
    for r in rows:
        lines.append(f"| {r['item']} | {r['price']:.2f} |")
    return '\n'.join(lines)

def on_upload(image):
    """Detect items in the uploaded photo and populate the fixed item slots fresh - always a
    full replace for THIS photo, never accumulating across photos (accumulation happens via
    the Add to Cart button instead, which keeps each event's output batch small)."""
    if image is None:
        blanks = [gr.update(visible=False, value=None) for _ in range(MAX_ITEMS * 3)]
        return [None, {}, ''] + blanks
    annotated, crops, preds = detect_and_classify(image)
    n = min(len(crops), MAX_ITEMS)
    photo_state = {'crops': crops[:n], 'preds': preds[:n]}
    updates = []
    for i in range(MAX_ITEMS):
        if i < n:
            updates.append(gr.update(visible=True, value=crops[i]))
            updates.append(gr.update(visible=True, value=preds[i], choices=classes))
            updates.append(gr.update(visible=True, value=prices.get(preds[i], 1.0)))
        else:
            updates.append(gr.update(visible=False, value=None))
            updates.append(gr.update(visible=False, value=None))
            updates.append(gr.update(visible=False, value=None))
    photo_msg = f'{n} item(s) detected in this photo - review below, then Add to Cart.'
    return [annotated, photo_state, photo_msg] + updates

def on_add_to_cart(cart, photo_state, *current_dropdown_and_price):
    """Add whatever's currently shown in the item slots (for the current photo) into the cart."""
    crops = photo_state.get('crops', [])
    preds = photo_state.get('preds', [])
    n = len(crops)
    cart = list(cart)
    if n == 0:
        msg = 'Nothing to add - upload a photo first.'
    else:
        current_labels = list(current_dropdown_and_price[:MAX_ITEMS])[:n]
        current_prices = list(current_dropdown_and_price[MAX_ITEMS:2 * MAX_ITEMS])[:n]
        for crop, original, label, price in zip(crops, preds, current_labels, current_prices):
            final_label = label if label else original
            final_price = float(price) if price is not None else prices.get(final_label, 1.0)
            cart.append({'crop': crop, 'original_pred': original, 'label': final_label, 'price': final_price})
        msg = f'{len(cart)} item(s) in cart.'
    total = sum(e['price'] for e in cart)
    cart_md = format_items_markdown([{'item': e['label'], 'price': e['price']} for e in cart])
    return cart, {}, cart_md, f'{total:.2f} {currency}', msg

def on_new_cart():
    """Clear the cart and the last checkout's receipt to start over."""
    return [], {}, format_items_markdown([]), f'0.00 {currency}', '', format_items_markdown([]), ''

def on_confirm(cart, staff_id):
    """Finalize the cart: save any label corrections, persist any price overrides, build the receipt, then clear the cart."""
    if not cart:
        return cart, gr.update(), gr.update(), format_items_markdown([]), 'Cart is empty - add at least one item first.'
    preds = [e['original_pred'] for e in cart]
    proposed_labels = [e['label'] for e in cart]
    proposed_prices = [e['price'] for e in cart]
    authorized = (staff_id == STAFF_PIN)
    final_labels, final_prices, label_changed, price_changed, rejected = resolve_checkout_changes(
        preds, proposed_labels, proposed_prices, prices, authorized)
    saved = 0
    catalog_updated = 0
    for entry, final_label, changed_l in zip(cart, final_labels, label_changed):
        if changed_l:
            append_correction(b.root, entry['crop'], final_label)
            saved += 1
    for final_label, final_price, changed_p in zip(final_labels, final_prices, price_changed):
        if changed_p:
            persist_catalog_price(b.root, final_label, final_price)
            prices[final_label] = final_price
            catalog_updated += 1
    total = basket_total([{'fine': f} for f in final_labels], prices, default=1.0)
    rows = [{'item': f, 'price': p} for f, p in zip(final_labels, final_prices)]
    note = ''
    if any(rejected):
        note = ' Staff ID required to change an item or price - your changes were not applied.'
    msg = f'Checked out {len(final_labels)} item(s), total {total:.2f} {currency}. Saved {saved} correction(s), updated {catalog_updated} catalog price(s).{note}'
    return [], format_items_markdown([]), f'0.00 {currency}', format_items_markdown(rows), msg

with gr.Blocks(theme=gr.themes.Soft(), title='SmartCart self-checkout v2') as demo:
    gr.Markdown('# SmartCart self-checkout (v2)\nUpload a counter photo, review the detected item(s) below, click **Add to Cart**, then repeat for more photos before confirming.')
    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type='pil', label='Counter photo')
            image_out = gr.Image(label='Detected items')
        with gr.Column():
            photo_state = gr.State({})
            cart_state = gr.State([])
            crop_images, label_dropdowns, price_boxes = [], [], []
            for i in range(MAX_ITEMS):
                with gr.Row():
                    ci = gr.Image(visible=False, label=f'Item {i+1}', height=100, width=100)
                    dd = gr.Dropdown(visible=False, choices=classes, allow_custom_value=True,
                                      label=f'Item {i+1} product (pick or type a new one)')
                    pb = gr.Number(visible=False, label='Price ($)')
                crop_images.append(ci); label_dropdowns.append(dd); price_boxes.append(pb)
            photo_msg = gr.Textbox(label='This photo', interactive=False)
            add_to_cart_btn = gr.Button('Add to Cart')
            gr.Markdown('**Cart**')
            cart_display = gr.Markdown('_(empty)_')
            running_total = gr.Textbox(label='Running total', interactive=False)
            staff_id_box = gr.Textbox(label='Staff ID (only needed if you changed or priced an item)', type='password')
            with gr.Row():
                confirm_btn = gr.Button('Confirm & Checkout')
                new_cart_btn = gr.Button('New Cart')
            gr.Markdown('**Receipt**')
            receipt_table = gr.Markdown('_(empty)_')
            confirm_msg = gr.Textbox(label='Result', interactive=False)

    image_in.change(fn=on_upload, inputs=[image_in],
                     outputs=[image_out, photo_state, photo_msg] + [c for triple in zip(crop_images, label_dropdowns, price_boxes) for c in triple])
    for dd, pb in zip(label_dropdowns, price_boxes):
        dd.change(fn=sync_price, inputs=[dd], outputs=[pb])
    add_to_cart_btn.click(fn=on_add_to_cart, inputs=[cart_state, photo_state] + label_dropdowns + price_boxes,
                           outputs=[cart_state, photo_state, cart_display, running_total, photo_msg])
    new_cart_btn.click(fn=on_new_cart, inputs=[],
                        outputs=[cart_state, photo_state, cart_display, running_total, photo_msg, receipt_table, confirm_msg])
    confirm_btn.click(fn=on_confirm, inputs=[cart_state, staff_id_box],
                       outputs=[cart_state, cart_display, running_total, receipt_table, confirm_msg])

def free_port(port):
    """Kill whatever process is currently listening on `port`, if any, so demo.launch()
    doesn't fail with 'Cannot find empty port' from a leftover server that never shut down."""
    try:
        import psutil
    except ImportError:
        print(f'psutil not available - if port {port} is busy, kill the process manually and re-run this cell.')
        return
    killed = 0
    for conn in psutil.net_connections(kind='inet'):
        if conn.laddr and conn.laddr.port == port and conn.pid:
            try:
                proc = psutil.Process(conn.pid)
                name = proc.name()
                proc.kill()
                print(f'killed stale process {conn.pid} ({name}) on port {port}')
                killed += 1
            except (psutil.NoSuchProcess, psutil.AccessDenied) as e:
                print(f'could not kill process {conn.pid} on port {port}: {e}')
    if not killed:
        print(f'port {port} was already free')

os.environ['SC_LAUNCH_GRADIO'] = '1'

if os.environ.get('SC_LAUNCH_GRADIO', '0') == '1':
    free_port(7861)
    demo.launch(share=True, server_port=7861)
else:
    print('Set SC_LAUNCH_GRADIO=1 (in Colab) to launch the shareable app.')

port 7861 was already free


C:\Users\drama\AppData\Local\Temp\ipykernel_27468\365490517.py:167: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='SmartCart self-checkout v2') as demo:


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


c:\Users\drama\Desktop\^SIT- SNAIC\.venv313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\drama\Desktop\^SIT- SNAIC\.venv313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\drama\Desktop\^SIT- SNAIC\.venv313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\drama\Desktop\^SIT- SNAIC\.venv313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead

## Showcase + leaderboard

Drop your best counter photo into the app and screenshot the basket. Post your validation accuracy (Day 3) and rare-class lift (Day 4) to the class leaderboard — we celebrate the biggest honest lift, not just the highest single number.

## Week recap / where next

You built the full deployable-vision loop: **acquire -> localize -> recognize -> diagnose -> augment -> assemble -> ship**. From here, see `deep_dive/` for the optional generative-augmentation stack, true open-set recognition, and on-device quantization-aware training. Same bundle contract, deeper each step.